# 02 — Backbone instantiation at tiny size

This notebook confirms that every registered backbone can be instantiated at a small width and produce a tensor of the contracted output shape from a single fixed synthetic input. This is the structural precondition for the benchmark: a backbone that cannot map `(N, C_in, H, W)` to `(N, C_out, H, W)` cannot be compared fairly against the others.

Modules exercised: `models/__init__.py` (`get_model`) and `pipelines/benchmark_pipeline/sizing.py` (`WidthScaler`, `_IMAGE_SIZE_MODELS`) to obtain reduced-width configurations identical in spirit to the pipeline's.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## Fixed synthetic input

A single seeded batch is created once and reused for every architecture. Using the same tensor across all backbones is what makes the shape comparison meaningful: only the model changes.

In [ ]:
from models import get_model, CONFIG_REGISTRY
from pipelines.benchmark_pipeline.sizing import WidthScaler, _IMAGE_SIZE_MODELS

BATCH        = 2
IN_CHANNELS  = 9
OUT_CHANNELS = 15
IMAGE_SIZE   = 32
SCALE        = 0.15

x = torch.randn(BATCH, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
print("fixed input shape :", tuple(x.shape))
print("expected output   :", (BATCH, OUT_CHANNELS, IMAGE_SIZE, IMAGE_SIZE))

## Instantiate, forward, and record

Each backbone is built at width scale 0.15 through the same `WidthScaler` the pipeline uses, switched to evaluation mode, and run once under `no_grad`. We record the parameter count, the realised output shape, and whether it matches the contract.

In [ ]:
scaler  = WidthScaler()
records = []

for name in CONFIG_REGISTRY:
    config    = scaler.scaled_config(name, SCALE)
    overrides = {"in_channels": IN_CHANNELS, "out_channels": OUT_CHANNELS}
    if name in _IMAGE_SIZE_MODELS:
        overrides["image_size"] = IMAGE_SIZE

    model, _ = get_model(name, config=config, **overrides)
    model.eval()

    with torch.no_grad():
        y = model(x)

    out_shape = tuple(y.shape)
    expected  = (BATCH, OUT_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

    records.append({
        "name"       : name,
        "parameters" : sum(p.numel() for p in model.parameters()),
        "out_shape"  : out_shape,
        "ok"         : out_shape == expected,
    })
    del model, y

for r in records:
    flag = "ok" if r["ok"] else "MISMATCH"
    print(f"{r['name']:22s} params={r['parameters']:>8,}  out={str(r['out_shape']):20s} {flag}")

assert all(r["ok"] for r in records), "at least one backbone broke the shape contract"
print("\nall backbones satisfy the output-shape contract")

## Visual confirmation of the shape contract

The grid below colours one cell per architecture: green when the realised output shape equals the contracted shape, red otherwise. A fully green grid is the quick visual proof that every backbone is benchmark-ready at tiny size.

In [ ]:
names   = [r["name"] for r in records]
ok_grid = np.array([1.0 if r["ok"] else 0.0 for r in records]).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(3.4, 9))
ax.imshow(ok_grid, cmap=plt.cm.RdYlGn, vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xticks([])
ax.set_title("output shape matches contract")
ax.grid(False)
fig.tight_layout()
plt.show()

## Expected visual outcome

Every printed row reports `ok` and the assertion passes. The vertical grid is uniformly green, confirming that all twenty-one backbones instantiate at reduced width and map the fixed synthetic input to the expected `(N, out_channels, H, W)` tensor.